# Interfaces → Duck Typing, Protocols, ABCs & Dunder Methods

## What you'll practice

Python asks "can it do the thing?" rather than "does it implement the interface?". `typing.Protocol` documents a shape for type checkers
without inheritance; `abc.ABC` enforces implementation when you want a hard contract; dunder methods (`__len__`, `__iter__`, `__add__`,
`__call__`) plug your objects into the language itself.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/04_java_to_python/04_interfaces_vs_duck_typing.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · If it has .size(), it's a source

**Implement:** `total_size(sources) -> int`

Sum `source.size()` across objects of **unrelated** classes (no common base class). Objects without a `size` method are skipped
(use `getattr(obj, "size", None)` and `callable`).

**Input:** `sources`

**Expected output:** `int`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_total_size` in `tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py`.

**Check:** `pytest tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py -k test_total_size`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · A Sink protocol with interchangeable implementations

**Implement:** `class Sink(Protocol)`, `class ListSink`, `class CsvSink`, `run_export(records, sink, batch_size=2) -> int`

Define `Sink(Protocol)` with `write(self, records: list[dict]) -> int` (returns rows written). Implement `ListSink` (stores rows in `.rows`)
and `CsvSink(path, fieldnames)` (appends rows to a CSV, writing the header only once even across multiple `write` calls). `run_export` writes records
in batches of `batch_size` and returns the total written. None of the sinks inherit from `Sink` — they just match its shape.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_list_sink, test_csv_sink_header_once, test_any_object_with_write_works` in `tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py`.

**Check:** `pytest tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py -k "test_list_sink or test_csv_sink_header_once or test_any_object_with_write_works"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Abstract base classes for validators

**Implement:** `class Validator(ABC)`, `class RequiredFields`, `class RangeCheck`, `validate_all(record, validators) -> list[str]`

`Validator(ABC)` has an abstract `validate(record) -> list[str]` and a concrete `__call__` that delegates to `validate` (so validators can be used like
functions). `RequiredFields(*fields)` reports `"<field> is required"` for missing/None/blank fields. `RangeCheck(field, lo, hi)` reports
`"<field> out of range [lo, hi]"` when present and outside range (missing → no error; non-numeric → `"<field> is not numeric"`).
`validate_all` concatenates errors from all validators. Instantiating `Validator` directly must fail with `TypeError`.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_validator_abc, test_validators` in `tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py`.

**Check:** `pytest tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py -k "test_validator_abc or test_validators"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Value objects through dunder methods

**Implement:** `class Money`

`Money(amount, currency)` stores `amount` as `Decimal` (accept str/int/Decimal; floats → `TypeError` to avoid precision bugs) and upper-case currency.
Support: `+` and `-` (same currency only, else `ValueError`), `* int|Decimal`, `==` and `<`/`<=`/`>`/`>=` (same currency; use
`functools.total_ordering`), `hash` (usable in sets/dict keys), `repr` → `Money('10.50', 'INR')`, `str` → `INR 10.50` (2 decimals),
and `sum([...], start=Money("0", "INR"))` **as well as plain** `sum(moneys)` (implement `__radd__` treating `0` as identity). Instances are immutable.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_money_arithmetic, test_money_compare_hash_repr, test_money_errors` in `tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py`.

**Check:** `pytest tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py -k "test_money_arithmetic or test_money_compare_hash_repr or test_money_errors"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Strategy pattern → plain callables · 🧩 Mini-project

**Implement:** `best_price(order: dict, strategies: list) -> tuple[float, str]`, `class TieredDiscount`, `percent_off(pct: float, min_total: float = 0)`

Java: `interface DiscountStrategy { BigDecimal apply(Order o); }` + three implementing classes + a `DiscountEngine`.
Python: a strategy is **any callable** `order -> discount_amount`. Implement:
- `percent_off(pct, min_total=0)` → returns a function giving `total × pct/100` when `order["total"] >= min_total`, else 0
- `TieredDiscount(tiers)` — a class with `__call__`; `tiers` is `[(threshold, flat_discount), ...]` (any order); picks the largest threshold `<= total`
- `best_price(order, strategies)` → `(final_price, strategy_name)` choosing the **largest** discount (never below 0 final price; ties → first strategy).
  Strategy name: `getattr(s, "__name__", type(s).__name__)`; give the function returned by `percent_off` a name like `percent_off_10`.
  No strategies or all-zero → `(total, "none")`.

**Input:** `order: dict, strategies: list`

**Expected output:** `tuple[float, str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
best_price({'total': 100}, [])
# -> (100, 'none')
```

**Check:** `pytest tests/java_to_python/test_04_04_interfaces_vs_duck_typing.py -k "test_percent_off_and_names or test_tiered or test_best_price"`

**YOUR CODE ↓**